# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [1]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 3: Poisson Equation and the Symmetric Interior Penalty Flux 

## Symmteric Interior Penalty

### Problem statement

We consider the two-phase Poisson problem in $\Omega = \mathfrak{A} \ {\dot\cup} \ \mathfrak{I} \ {\dot\cup} \ \mathfrak{B}, \Omega \in R^2$:
$$ -\mu \Delta u = f(x,y) \quad \textrm{in}\ \Omega \setminus \mathfrak{I} $$

with piece-wise constant diffusion coefficients
>$$ \mu = \begin{cases} 
\mu_\mathfrak{A} &\textrm{ in } \mathfrak{A}\\
\mu_\mathfrak{B} &\textrm{ in } \mathfrak{B}
\end{cases} $$

and $f(x,y) \neq 0$ is an arbitrary function of $x$ and/or $y$. The corresponding jump condition is given by
>$$ \llbracket \mu \nabla u \rrbracket \cdot \vec{n}_{\mathfrak{I}} = 0 $$


Within this exercise, we are going to implement and investigate the Symmetric Interior Penalty discretization method (SIP) for the Laplace operator

$$a_{\text{sip}}(u,v)
= \int_{\Omega} \underbrace{\nabla u \cdot \nabla v}_{\text{Volume\ term}}dV
  - \oint_{(\Gamma \setminus \Gamma_{N }) \cup \mathfrak{I}} \underbrace{
        \{\nabla u\} \cdot n_{\Gamma} \llbracket v \rrbracket
     }_{\text{consistency term}} + \underbrace{
        \{\nabla v\} \cdot \vec{n}_{\Gamma} \llbracket u \rrbracket
     }_{\text{symmetry term}} dA
  + \oint_{(\Gamma \setminus \Gamma_{N}) \cup \mathfrak{I}} \underbrace{
       \eta \llbracket u \rrbracket \llbracket v \rrbracket
    }_{\text{penalty term}} dA$$

We assume Dirichlet boundary conditions $u = u_D$ on all boundaries, i.e $\Gamma = \Gamma_i \cup \Gamma_D$.

### Implementation of the SIP fluxes

We define the diffusion coefficients $\mu_\mathfrak{A}$ and $\mu_\mathfrak{A}$, the RHS function $f(x,y)$ and the Dirichlet value $u_D$ as global variables/delegates, which can be alteree later on.

In [2]:
static class ProblemMap { 

    public static double muA = 1.0;
    public static double muB = 1.0;

    public static Func<double[], double> RHSfunc = null;

    public static Func<double[], double> UDiriA = null;
    public static Func<double[], double> UDiriB = null;
}

First, we need a class in which the integrands for the separate phases are defined including the volume terms, inner edges (excluding the interface!) and boundary terms.

In [ ]:
public class SipLaplaceX_Bulk :
        BoSSS.Foundation.IVolumeForm, // volume integrals
        BoSSS.Foundation.IEdgeForm,   // edge integrals
        IEquationComponentCoefficient, // update of coefficients (e.g. length scales) required for penalty parameters 
        IEquationComponentSpeciesNotification   // switch coefficients based on species
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }
    /// The \code{TermActivationFlags} tell \BoSSS which kind of terms, 
    /// i.e. products of u, v, \nabla u, and \nabla v
    /// the VolumeForm(...) actually contains.
    /// This additional information helps to improve the performance.
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
    /// activation flags for the 'InnerEdgeForm(...)'
    public TermActivationFlags InnerEdgeTerms {
        get {
            //return TermActivationFlags.UxV | TermActivationFlags.GradUxV | TermActivationFlags.UxGradV;
            // if we do not care about performance, we can activate all terms.
            return (TermActivationFlags.AllOn);
        }
    }
    public TermActivationFlags BoundaryEdgeTerms {
       get {
            //return TermActivationFlags.UxV | TermActivationFlags.GradUxV | TermActivationFlags.UxGradV;
            return TermActivationFlags.AllOn;
        }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cj;

    double penalty_base; // base factor must be scaled by polynomial degree  

    /// The additional scaling of the penalty by polynomial degree 
    /// and in dependence of geometry can be obtained through 
    /// implementing the \code{IEquationComponentCoefficient} interface:
    public void CoefficientUpdate(CoefficientSet cs, int[] DomainDGdeg, int TestDGdeg) {
        int D = cs.GrdDat.SpatialDimension;
        double _D = D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice
        //Console.WriteLine("Setting penalty base factor for deg " + _p + " to " + penalty_base);

        cj = ((GridData)(cs.GrdDat)).Cells.cj; // inverse length scale (dimension is one-over-length, resp. area over volume)
    }     
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double sf_in         = cj[jCellIn];   // size factor, inverse length scales
        double eta           = penalty_base * sf_in * PenaltySafety;
        if(jCellOut >= 0) {
            double sf_out = cj[jCellOut];
            eta           = Math.Max(eta, penalty_base * sf_out * PenaltySafety);
        }
        
        return eta;
    }
    
    // Sets the parameters depending on the respective species.
    // In our case the diffusion coeffients and Dirichlet values will be set according to our global variables/delegates.
    public void SetParameter(string speciesName) {
        switch (speciesName) {
            case "A": muSpc = ProblemMap.muA; UDiri = ProblemMap.UDiriA; break;
            case "B": muSpc = ProblemMap.muB; UDiri = ProblemMap.UDiriB; break;
            default: throw new ArgumentException("Unknown species.");
        }
    }

    double muSpc;

    Func<double[], double> UDiri;

    /// The following functions cover the actual math.
    /// For any discretization of the Laplace operator, we have to specify:
    /// 
    /// - a volume integrand,
    /// - an edge integrand for inner edges, i.e. on $\Gamma_i$,
    /// - an edge integrand for boundary edges, i.e. on $\partial \Omega$.
    /// 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU, // the trial-function u (i.e. the function we search for) and its gradient
           double V, double[] GradV     // the test function; 
           ) {

        double acc = 0.0;       
        // == TODO == add the SIP volume form 

        return acc;
    }

    /// The integrand for the integral on the inner edges,
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
 
        double Acc = 0.0;
        // == TODO == add the consistency term : -({ \/u } [[ v ]])*Normal
        // == TODO == add the symmetry term: : -({ \/v } [[ u ]])*Normal
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        return muSpc * Acc;
    } 

    /// The integrand on boundary edges, i.e. on $\partial \Omega$, is
    ///  
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   +  \eta [[u]]  [[v]] .
    /// 
    /// For the boundary we have to consider the special definition for 
    /// the mean-value operator ${-}$ and the jump operator $[[-]]$ on the boundary.
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
 
        double eta = PenaltyFactor(inp.jCellIn, -1);
        
        double Acc = 0.0;
        // == TODO == add the consistency term: -({ \/u } [[ v ]])*Normale
        // == TODO == add the symmetry term: -({ \/v } [[ u ]])*Normale
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        return Acc;
    }
}

New we need to define the terms active on the interface $\mathfrak{I}$.

In [ ]:
public class SipLaplaceX_Interface : 
                    ILevelSetForm,  // interface integrals
                    ILevelSetEquationComponentCoefficient  // update of coefficients (e.g. length scales) required for penalty parameters 
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }

    public TermActivationFlags LevelSetTerms {
        get {
            return TermActivationFlags.UxV | TermActivationFlags.GradUxV | TermActivationFlags.UxGradV;
        }
    }

    public int LevelSetIndex {
        get { return 0; }
    }

    public string NegativeSpecies {
        get { return "A"; }
    }

    public string PositiveSpecies {
        get { return "B"; }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cjA;
    MultidimensionalArray cjB;

    double penalty_base; // base factor must be scaled by polynomial degree  

    public void CoefficientUpdate(CoefficientSet csA, CoefficientSet csB, int[] DomainDGdeg, int TestDGdeg) {
        cjA = ((GridData)(csA.GrdDat)).Cells.cj; 
        cjB = ((GridData)(csB.GrdDat)).Cells.cj; 

        double _p = DomainDGdeg.Max();
        double _D = csA.GrdDat.SpatialDimension;
        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes
            
        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr);
    }   
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double penaltySizeFactor_A = cjA[jCellIn];
        double penaltySizeFactor_B = cjB[jCellOut];

        double penaltySizeFactor = Math.Max(penaltySizeFactor_A, penaltySizeFactor_B);

        double scaledPenalty = penaltySizeFactor * PenaltySafety * penalty_base;
        if(scaledPenalty.IsNaNorInf())
            throw new ArithmeticException("NaN/Inf detected for penalty parameter.");
        return scaledPenalty;
    }    

    /// The integrand for the integral on the interface (inner edge),
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public virtual double InnerEdgeForm(ref CommonParams inp, 
        double[] uA, double[] uB, double[,] Grad_uA, double[,] Grad_uB,
        double vA, double vB, double[] Grad_vA, double[] Grad_vB) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
        double muA = ProblemMap.muA;
        double muB = ProblemMap.muB;    
 
        double Acc = 0.0;
        // == TODO == add the consistency term : -({ mu \/u } [[ v ]])*Normal
        // == TODO == add the symmetry term: : -({ mu \/v } [[ u ]])*Normal
        // == TODO == add the penalty term: mu*eta*[[u]]*[[v]]    

        return Acc;
    } 
    
}

The source term on the right-hand-side (RHS) will also be implemented as volume terms.

In [5]:
class RHSSource : IVolumeForm {

    public TermActivationFlags VolTerms => TermActivationFlags.V;

    public IList<string> ArgumentOrdering => new string[0];

    public IList<string> ParameterOrdering => new string[0];

    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double V, double[] GradV) {
        double rhsVal = ProblemMap.RHSfunc(cpv.Xglobal);
        return -rhsVal * V;
    }
}

### Evaluation of the Poisson operator in (pseudo) 1D

We consider the following problem in 1D:
$$
- \mu \Delta u = 2,\quad -1<x<1,\quad u(-1)=u(1)=0,
$$

where the interface $\mathfrak{I}$ is located at x = 0.0 with phase $\mathfrak{A}$ in $x < 0.0$ and phase $\mathfrak{B}$ in $x > 0.0$ 

The solutions for both phases is given by:
$$ 
u_{\mathfrak{A}, ex}(x) = -\frac{1}{\mu_{\mathfrak{A}}} x^2 + \frac{\mu_{\mathfrak{A}} - \mu_{\mathfrak{B}}}{\mu_{\mathfrak{A}}(\mu_{\mathfrak{A}} + \mu_{\mathfrak{B}})}x + \frac{2}{\mu_{\mathfrak{A}} + \mu_{\mathfrak{B}}} \\
u_{\mathfrak{B}, ex}(x) = -\frac{1}{\mu_{\mathfrak{B}}} x^2 + \frac{\mu_{\mathfrak{A}} - \mu_{\mathfrak{B}}}{\mu_{\mathfrak{B}}(\mu_{\mathfrak{A}} + \mu_{\mathfrak{B}})}x + \frac{2}{\mu_{\mathfrak{A}} + \mu_{\mathfrak{B}}}
$$ 

Since they are quadratic, we can represent the solutions **exactly** in a DG space of order 2.

First, we define the global variables and delegates.

In [6]:
double muA = 1.0;
double muB = 20.0;
ProblemMap.muA = muA;
ProblemMap.muB = muB;

In [7]:
Func<double[],double> RHS_pseudo1D = ((double[] X) => 2.0);
ProblemMap.RHSfunc = RHS_pseudo1D;

The implementation of the exact solution:

In [8]:
double a2 = -1.0 / muA;
double a1 = (muA - muB) / (muA * (muA + muB));
double a0 = 2.0 / (muA + muB);
Func<double[], double> uEx_A = (X => (a2 * X[0].Pow2() + a1 * X[0] + a0));

double b2 = -1.0 / muB;
double b1 = (muA / muB) * a1;
double b0 = a0;
Func<double[], double> uEx_B = (X => (b2 * X[0].Pow2() + b1 * X[0] + b0));

In [9]:
ProblemMap.UDiriA = uEx_A;
ProblemMap.UDiriB = uEx_B;

We define a grid with an uneven number of cells in $x$-direction and only one cell in $y$-direction.

In [10]:
int jCells = 3;
Grid2D grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1, 1, jCells + 1),   
                                        GenericBlas.Linspace(0, 2.0/jCells, 2));    // 1 cell  
GridData gdata2D = new GridData(grd2D);     

We need to define a level-set field, where the zero-set defines the location of the interface $\mathfrak{I}$. The level-set field is given as a standrad DG-Field.

In [11]:
LevelSet LevelSetField = new LevelSet(new Basis(gdata2D, 1), "Interface");   // line interface
LevelSetField.ProjectField((double[] X) => X[0]);

The `LevelSetTracker` manages the tracking of narrow-bands of the level-set and thier respective species domains. 
It is a perquisite for the memory management of cut-cell fields, such as the `XDGField`.

In [12]:
LevelSetTracker LsTrk = new LevelSetTracker(gdata2D, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] {"A", "B"}, LevelSetField);
LsTrk.UpdateTracker(0.0);

In [13]:
var Operator_SipLaplaceX = new XDifferentialOperatorMk2(1, 1, QuadOrderFunc.Linear(), LsTrk.SpeciesNames, "u", "lap"); 
Operator_SipLaplaceX.EquationComponents["lap"].Add(new SipLaplaceX_Bulk()); 
Operator_SipLaplaceX.EquationComponents["lap"].Add(new SipLaplaceX_Interface()); 
Operator_SipLaplaceX.EquationComponents["lap"].Add(new RHSSource());
Operator_SipLaplaceX.Commit();

We now want to calculate the residual after inserting the exact solution as well as a wrong solution. 

In [14]:
XDGBasis xdgBasis2D = new XDGBasis(LsTrk, 2);
XDGField u_ex = new XDGField(xdgBasis2D, "$u_{ex}$");
u_ex.GetSpeciesShadowField("A").ProjectField(uEx_A);
u_ex.GetSpeciesShadowField("B").ProjectField(uEx_B);

The implementation of a spurious, i.e. a wrong solution; we take the exact solution and add random values in each cell:

In [15]:
XDGField u_wrong      = new XDGField(xdgBasis2D, "$u_{wrong}$");
u_wrong.GetSpeciesShadowField("A").ProjectField(uEx_A);
u_wrong.GetSpeciesShadowField("B").ProjectField(uEx_B);
Random R         = new Random();
for(int j = 0; j < grd2D.GridData.Cells.NoOfLocalUpdatedCells; j++){
    double ujMean = u_wrong.GetMeanValue(j);
    ujMean += R.NextDouble();
    u_wrong.SetMeanValueAB(j, ujMean);
}

Evaluating the Laplace operator using the different solutions:

In [16]:
var Mapping2D   = new UnsetteledCoordinateMapping(xdgBasis2D);
XDGField Residual_eval = new XDGField(xdgBasis2D, "Residual_eval");
var ResidualNorm = new List<double>();
foreach(var u in new XDGField[] {u_ex, u_wrong}) {
    Residual_eval.Clear();
    Operator_SipLaplaceX.GetEvaluatorEx(LsTrk, new XDGField[] { u }, null, Mapping2D).Evaluate(1.0, 0.0, new CoordinateVector(Residual_eval));  // evaluate
    double ResiNorm = Residual_eval.L2Norm();
    ResidualNorm.Add(ResiNorm);
    Console.WriteLine("Residual for " + u.Identification + " = " + ResiNorm);  
}

Residual for $u_{ex}$ = 1.0495822580097619E-12
Residual for $u_{wrong}$ = 9564.056934646735


In [17]:
using NUnit.Framework;

In [18]:
Assert.LessOrEqual(ResidualNorm[0], 1e-10);
Assert.GreaterOrEqual(ResidualNorm[1], 1e-1);

In [19]:
DGField[] plotFields = new DGField[] { u_ex, Residual_eval };
Tecplot("plots/plot_HandsOn3_xdgPoisson1D",  plotFields);

Writing output file plots/plot_HandsOn3_xdgPoisson1D...
done.


### (pseudo) two-phase solution in 2D

We consider the one phase example (i.e. $\mu_{\mathfrak{A}} = \mu_{\mathfrak{B}} = 1.0$)
$$
    -\Delta u = \pi^2 \frac{(a_x^2 + a_y^2)}{4} \cos(a_x \frac{\pi x}{2}) \cos(a_y \frac{\pi y}{2}) 
      \quad \text{ on } 
      (x,y) \in (-1,1)^2
$$
and $u = 0$ on the boundary.
The exact solution is $u_{ex}(x,y) = \cos(a_x \frac{\pi x}{2}) \cos(a_x \frac{\pi y}{2})$, where $a_x$ and $a_y$ must be odd numbers
to comply with homogeneous boundary condition.

The interface may be set arbitrary. e.g. $\varphi(x,y) = x - y$.

Again we set the global variables and delegates

In [19]:
ProblemMap.muA = 1.0;
ProblemMap.muB = 1.0;

In [20]:
double ax = 1.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
double ay = 3.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
Func<double[], double> exRhs_2D = 
        (X => ((ax.Pow2() + ay.Pow2())/4.0)*Math.PI.Pow2()
             *Math.Cos( X[0]*ax*Math.PI*0.5 )*Math.Cos( X[1]*ay*Math.PI*0.5 )); 
Func<double[], double> exSol_2D = 
        (X => Math.Cos(X[0]*ax*Math.PI*0.5)*Math.Cos(X[1]*ay*Math.PI*0.5));

In [21]:
ProblemMap.RHSfunc = exRhs_2D;
ProblemMap.UDiriA  = exSol_2D;
ProblemMap.UDiriB  = exSol_2D;

We define a two-dimensional grid:

In [22]:
var grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,21), 
                                    GenericBlas.Linspace(-1,1,16));

We set up the `LevelSetTracker`

In [23]:
LevelSet LevelSetField = new LevelSet(new Basis(grd2D, 1), "Interface"); 
LevelSetField.ProjectField((double[] X) => X[0] - X[1]);
LevelSetTracker LsTrk = new LevelSetTracker(grd2D.GridData, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] {"A", "B"}, LevelSetField);
LsTrk.UpdateTracker(0.0);

and the corresponding `XDGBasis` and `UnsetteledCoordinateMapping`

In [33]:
var xdgBasis2D = new XDGBasis(LsTrk, 5);
var Mapping2D = new UnsetteledCoordinateMapping(xdgBasis2D);

We check our discretization once more in 2D; the residual should be low,
but not exactly (resp. up to $10^{-12}$) since the solution is not 
polynomial and cannot be fulfilled exactly.

In [34]:
using ilPSP.LinSolvers;

In [35]:
XDGField u = new XDGField(xdgBasis2D,"u");
u.ProjectField(exSol_2D);

In [27]:
DGField[] plotFields = new DGField[] { u, LevelSetField };
Tecplot("plots/plot_HandsOn3_xdgPoisson2D",  plotFields);

Supersampling 4 requested, but limiting to 3 because higher values would very likely exceed this computers memory.
Note: Higher supersampling values are supported by external plot application, or by using e.g. the Tecplot class directly.
Writing output file plots/plot_HandsOn3_xdgPoisson2D...
done.


In [36]:
BlockMsrMatrix Matrix_xSIP = new BlockMsrMatrix(Mapping2D);
double[] Affine = new double[Mapping2D.LocalLength];
Operator_SipLaplaceX.GetMatrixBuilder(LsTrk, Mapping2D, null, Mapping2D).ComputeMatrix(Matrix_xSIP, Affine);

XDGField Residual = new XDGField(xdgBasis2D,"Residual");
Residual.CoordinateVector.AccV(-1.0, Affine);
Matrix_xSIP.SpMV(-1.0, u.CoordinateVector, 1.0, Residual.CoordinateVector);
Console.WriteLine("Residual L2 norm: " + Residual.L2Norm());

Residual L2 norm: 0.021145548385326958


In [30]:
//== Task == Above, how could you decrease the Residual L2 norm?

We also check that the matrix is symmetric:

In [37]:
var checkMatrix = Matrix_xSIP - Matrix_xSIP.Transpose();
checkMatrix.InfNorm()

5.832928584581509E-10

In [38]:
Assert.LessOrEqual(checkMatrix.InfNorm(), 1e-8);

### Convergence study

In [33]:
double[] Resolutions = new double[] { 2, 4, 8, 16, 32, 64 };
List<double> L2Errors  = new List<double>();
int cnt = 0;
foreach(int Res in Resolutions) {
    cnt++;
    var grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
                                      GenericBlas.Linspace(-1,1,(int)Res + 1));

    LevelSet LevelSetField = new LevelSet(new Basis(grd2D, 1), "Interface");   // line interface
    LevelSetField.ProjectField((double[] X) => X[0] - X[1]);
    LevelSetTracker LsTrk = new LevelSetTracker(grd2D.GridData, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] {"A", "B"}, LevelSetField);
    LsTrk.UpdateTracker(0.0);

    var gdata2D = new GridData(grd2D);
    var XDGBasisOn2D = new XDGBasis(LsTrk, 2);
    var Mapping2D  = new UnsetteledCoordinateMapping(XDGBasisOn2D);
 
    XDGField uEx = new XDGField(
           new XDGBasis(LsTrk, XDGBasisOn2D.Degree*2),
           "Error");
    uEx.ProjectField(exSol_2D);
 
    BlockMsrMatrix Matrix_xSIP = new BlockMsrMatrix(Mapping2D);
    double[] Affine = new double[Mapping2D.LocalLength];
    Operator_SipLaplaceX.GetMatrixBuilder(LsTrk, Mapping2D, null, Mapping2D).ComputeMatrix(Matrix_xSIP, Affine);
 
    XDGField u = new XDGField(XDGBasisOn2D, "u");
    XDGField RHS = new XDGField(XDGBasisOn2D, "RHS");
    RHS.CoordinateVector.AccV(-1.0, Affine);
    Matrix_xSIP.Solve_Direct(u.CoordinateVector, 
                                  RHS.CoordinateVector);
    var Error = uEx.CloneAs();
    Error.AccLaidBack(-1.0, u);
    L2Errors.Add(Error.L2Norm());

    
    //Tecplot("ConvStudy-" + cnt, uEx, u_posdef, u_indef); // activate this line for plotting!
    
    Console.WriteLine(L2Errors.Last().ToString("0.#E-00"));
}

6.2E-01
1.5E-01
1.7E-02
1.7E-03
1.8E-04
2.2E-05


In [34]:
var plt = new Plot2Ddata();
plt.AddDataGroup("xSIP", Resolutions, L2Errors);

/// A double-logarithmic scale is used:
plt.LogX = true;
plt.LogY = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Red, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);

// Show!
plt.PlotNow()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 xSIP 
 
 
 xSIP

Finally, we are going to compute the convergence rate of the SIP
discretization.

We compute the slope of the log-log plot:

In [35]:
double dk = Resolutions.LogLogRegressionSlope(L2Errors);
dk

-3.0362125863426583

In [36]:
Assert.LessOrEqual(dk, -2.9);